# DockQ打分
对AF3的预测结果运行DockQ，输出docking_results.csv

AF3预测结构链: A(受体), B(肽链)
参考天然结构(DockQ格式 `MODEL:NATIVE`): `AB:{non_L}L`
- A → {non_L} (受体)
- B → L (肽链)


In [1]:
import os
import subprocess
from pathlib import Path
import pandas as pd
import json
from multiprocessing import Pool
from Bio.PDB import PDBParser

DOCKQ = '/home/junjiechen/miniforge3/envs/dockq/bin/DockQ'
BASE = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/esmfold2'
METHODS = [
    # 'PepSet_dimer_output_esmfold2',
    # 'PepSet_dimer_output_esmfold2_fast',
    'PepSet_dimer_output_esmfold2_msa_nonpairing',
    'PepSet_dimer_output_esmfold2_fast_loop10_samplingsteps68'
]

REF_DIR = f'{BASE}/PepSet_dimer'
OUT_BASE = f'{BASE}/predict'
DOCKQ_OUT = './DockQ'
os.makedirs(DOCKQ_OUT, exist_ok=True)

parser = PDBParser(QUIET=True)

In [2]:
# 生成DockQ命令列表
def get_native_chains(ref_pdb):
    """读取ref_pdb，返回 (non_L_chain, L_chain)"""
    structure = parser.get_structure('ref', ref_pdb)
    chain_ids = [ch.id for ch in structure[0]]
    non_l = [c for c in chain_ids if c != 'L']
    if not non_l:
        raise ValueError(f"No non-L chain found in {ref_pdb}")
    return non_l[0], 'L'


pdbs = [pdb.split('.')[0] for pdb in sorted(os.listdir(REF_DIR)) if pdb.endswith('.pdb')]
with open('./DockQ/dockq_2.list', 'w') as f:
    for method in METHODS:
        for pdb in pdbs:
            ref_pdb = f'{REF_DIR}/{pdb}.pdb'
            non_l_chain, l_chain = get_native_chains(ref_pdb)            
            pred_dir = f'{OUT_BASE}/{method}/{pdb}'
            # if not os.path.isdir(pred_dir):
            #     print(f"Warning: {pred_dir} does not exist, skipping.")
            #     continue
            for seed in ['42', '43', '44']:
                for id in range(5):
                    pred_pdb = f'{pred_dir}/seed-{seed}-sample-{id}/complex_with_occupancy.cif'   
                    cmd = f'DockQ {pred_pdb} {ref_pdb} --short --mapping AB:{non_l_chain}L'
                    f.write(cmd + '\n')


# 解析DockQ输出

In [4]:
# 解析DockQ输出为DataFrame（支持method从model_path中提取）
def parse_dockq_output():
    """从合并的DockQ输出文件解析结果，自动从model_path提取method"""
    out_file = f'{DOCKQ_OUT}/dockq_2.out'
    
    if not os.path.exists(out_file):
        raise FileNotFoundError(f"DockQ output file not found: {out_file}")
    
    rows = []
    with open(out_file, 'r') as f:
        lines = f.readlines()
    
    for i, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) < 20:
            continue  # 跳过 "Total DockQ over..." 等非数据行
        try:
            dockq_score = float(parts[1])
            irmsd = float(parts[3])
            lrmsd = float(parts[5])
            fnat = float(parts[7])
            
            # 从model_path中提取method、pdb、seed、sample
            model_path = parts[16]
            native_path = parts[20]
            
            # path: .../original_result/{method}/output/{pdb}/seed-{seed}-sample-{sample}/{pdb}_seed-{seed}_sample-{sample}_model.cif
            # 提取method: 找到 /original_result/ 后的第一个目录名
            method = model_path.split('/esmfold2/predict/')[1].split('/')[0]
        
            native_name = native_path.split('/')[-1].replace('.pdb', '')
            
            seed_sample = model_path.split('/')[-2]  # seed-42-sample-0
            seed = seed_sample.split('-')[1]  # 42
            sample = seed_sample.split('-')[3]  # 0
            
            pdb = native_name
            
            
            rows.append({
                'method': method,
                'complex': native_name,
                'ref_pdb': pdb,
                'seed': seed,
                'id': sample,
                'dockq_score': dockq_score,
                'fnat': fnat,
                'lrmsd': lrmsd,
                'irmsd': irmsd,
            })
        except (ValueError, IndexError):
            continue
    
    return pd.DataFrame(rows) if rows else None

# 解析合并的DockQ输出并保存
df_all = parse_dockq_output()
if df_all is not None:
    print(f'Total: {len(df_all)} rows, {df_all["method"].nunique()} methods')
    for method_name in METHODS:
        df_method = df_all[df_all['method'] == method_name].copy()
        if len(df_method) > 0:
            os.makedirs(f'{DOCKQ_OUT}/results', exist_ok=True)
            df_method.to_csv(f'{DOCKQ_OUT}/results/{method_name}_docking_results.csv', index=False)
            print(f'  {method_name}: {len(df_method)} rows, {df_method["complex"].nunique()} complexes')
        else:
            print(f'  {method_name}: no data')
else:
    print('No results found (run DockQ first)')

Total: 5010 rows, 2 methods
  PepSet_dimer_output_esmfold2_msa_nonpairing: 2490 rows, 166 complexes
  PepSet_dimer_output_esmfold2_fast_loop10_samplingsteps68: 2520 rows, 168 complexes
